<div style='font-size:14px'>So now we haven't yet jumped into exploring and analyzing the job_skills column that has that list of skills and that's because there's a slight problem with this it's not contained as a list inside of our data frame right now its being annotated as a string luckily we have an apply method for df where we can actually go through and apply something to that column to clean it up  

In [3]:
import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset
import numpy as np

# Loading Data set
dataset= load_dataset('lukebarousse/data_jobs')
df = dataset['train'].to_pandas()

# cleanup
df['job_posted_date']=pd.to_datetime(df['job_posted_date'])

In [5]:
df[pd.notna(df['salary_year_avg'])]['salary_year_avg']

28        109500.0
77        140000.0
92        120000.0
100       228222.0
109        89000.0
            ...   
785624    139216.0
785641    150000.0
785648    221875.0
785682    157500.0
785692    157500.0
Name: salary_year_avg, Length: 22003, dtype: float64

<div style='font-size:14px'>What we want to do with this prblm is we want to calculate the projected salary for the next year and what we can do is we're going to apply what inflation is right now of arnd 3%, jst basically take these above values nd multiply it times well 1.03 to get to see what would be 3% higher next year so 

<div style='font-size:14px'>the documentation which we open on browser of pandas can be viewed here also with help

In [7]:
help(df.apply)

Help on method apply in module pandas.core.frame:

apply(func: 'AggFuncType', axis: 'Axis' = 0, raw: 'bool' = False, result_type: "Literal['expand', 'reduce', 'broadcast'] | None" = None, args=(), by_row: "Literal[False, 'compat']" = 'compat', engine: "Literal['python', 'numba']" = 'python', engine_kwargs: 'dict[str, bool] | None' = None, **kwargs) method of pandas.core.frame.DataFrame instance
    Apply a function along an axis of the DataFrame.
    
    Objects passed to the function are Series objects whose index is
    either the DataFrame's index (``axis=0``) or the DataFrame's columns
    (``axis=1``). By default (``result_type=None``), the final return type
    is inferred from the return type of the applied function. Otherwise,
    it depends on the `result_type` argument.
    
    Parameters
    ----------
    func : function
        Function to apply to each column or row.
    axis : {0 or 'index', 1 or 'columns'}, default 0
        Axis along which the function is applied:
 

<div style='font-size:14px'>now apply method has a few parameters 1st we will focus on function. A function is passed telling that these instructions should be applied on the data frame.

In [14]:
df_salary=df[pd.notna(df['salary_year_avg'])].copy()

In [ ]:
def projected_salary(salary):
    return salary * 1.03
 
df_salary['salary_year_inflated'] = df_salary['salary_year_avg'].apply(projected_salary)

df_salary [['salary_year_avg', 'salary_year_inflated']]

<div style='font-size:14px'>now this project_salary funct is pretty short and for providing this apply method a function, we could use something like an anonymous function.so lets try writting this again with anonymous function

######
##### anonymous function:
An anonymous function is a function definition without a specific name, designed for short-term, one-time use or passed as an argument to another function (callback). Known as lambda functions (Python/MATLAB) or function expressions (JavaScript), they are concise, often defined inline, and usually assigned to variables for immediate execution or mapping/filtering data. 

In [16]:
df_salary['salary_year_inflated'] = df_salary['salary_year_avg'].apply(lambda a: a*1.03)

df_salary [['salary_year_avg', 'salary_year_inflated']]

,salary_year_avg,salary_year_inflated
28,109500.0,112785.00
77,140000.0,144200.00
92,120000.0,123600.00
100,228222.0,235068.66
109,89000.0,91670.00
...,...,...
785624,139216.0,143392.48
785641,150000.0,154500.00
785648,221875.0,228531.25
785682,157500.0,162225.00


<div style='font-size:14px'>we can actually rewrite this all without apply method and lambda function as this is simple case of multiplying but this was mainly to show intro to this method. now lets apply in job_skills

In [22]:
type(df['job_skills'][1])

str

<div style='font-size:14px'>this cannot be converted into list using to_list cz str doesn't support to_list. Also theres a method of wrapping any str in list like list(df['job_skills'][1]) this converts str to list but it will take each  charachter of str and make it a list.Instead we will go inside python stndrd lib and use the module of ast which stnds for abstract syntax tree with this you provide a node or a string so in our case we're going to provide that string to it and it basically goes through and explains that it converts it to the container data type in our case a list back to it what it was supposed to be 

In [32]:
import ast

# ast.literal_eval(df['job_skills']) can't paas this command like this it will give error to pass it without index we need to use apply
ast.literal_eval(df['job_skills'][1])
# type(ast.literal_eval(df['job_skills'][1]))

['r', 'python', 'sql', 'nosql', 'power bi', 'tableau']

<div style='font-size:14px'>now lets clean up this column by applying m]apply function

In [ ]:
def clean_list(skill_list):
    return ast.literal_eval(skill_list)

df['job_skills'] = df['job_skills'].apply(clean_list)

<div style='font-size:14px'>this above code will give error as literal eval expect strings and this job_skill column also contain none value which is not a string so these nones will through error so we need to add iff condition in our function

In [ ]:
def clean_list(skill_list):
    if pd.notna(skill_list):
        return ast.literal_eval(skill_list)

df['job_skills'] = df['job_skills'].apply(clean_list)


In [ ]:
df['job_skills'] = df['job_skills'].apply(lambda skill_list : ast.literal_eval(skill_list) if pd.notna(skill_list) else skill_list)

In [40]:
type(df['job_skills'][2])

list

<div style='font-size:14px'>now we assumed an inflation of 3% lets assume for job role of sr. analysts and all the inflamation is 5% higher so we will use now axis parameter of apply in which 0 applies func to each column and 1 applies to row

<div style='font-size:14px'>now till now we were applying the apply method to a column now we will apply it to a row

In [ ]:
help(df.apply)

In [47]:
def projected_salary(row):
    if 'Senior' in row['job_title_short']:
        return 1.05 * row['salary_year_avg']
    else:
        return 1.03 * row['salary_year_avg']

df_salary['salary_year_inflated']=df_salary.apply(projected_salary, axis=1)

df_salary[['job_title_short','salary_year_avg', 'salary_year_inflated']]

,job_title_short,salary_year_avg,salary_year_inflated
28,Data Scientist,109500.0,112785.00
77,Data Engineer,140000.0,144200.00
92,Data Engineer,120000.0,123600.00
100,Data Scientist,228222.0,235068.66
109,Data Analyst,89000.0,91670.00
...,...,...,...
785624,Data Engineer,139216.0,143392.48
785641,Data Engineer,150000.0,154500.00
785648,Data Scientist,221875.0,228531.25
785682,Data Scientist,157500.0,162225.00


In [ ]:
df_salary['salary_year_inflated']=df_salary.apply(lambda row: 1.05 * row['salary_year_avg'] if 'Senior' in row['job_title_short'] else 1.03 * row['salary_year_avg'], axis=1)
